# W12｜Capstone / Production ChatBI System Design｜教学版

**建议学习时间：4h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射复盘，不开新课程 | 只收口与讲清系统 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 做 Agent Architecture Review | State/Tool/Checkpoint/Eval 边界清晰 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 注入 3 个故障 | worker crash / duplicate / LLM timeout | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 完成 Demo + 架构图 + README | 从问题到图表/答案端到端可演示 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 准备 10 分钟 Interview Story | 不用 AI 可讲 trade-off 与 failure mode | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜三种 Failure Injection
至少真实验证：duplicate POST、worker crash、LLM/tool timeout。

### Task 4｜README 参考结构
Problem / Architecture / Request Flow / Agentic-vs-Deterministic / Reliability / Observability & Eval / How to Run / Demo / Failure Tests / ADRs。

### Task 5｜10 分钟 Story
1min Problem；2min Architecture；2min AI Boundary；2min Production Reliability；1min Eval/Trace；2min Trade-offs/Next Step。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 能独立讲清端到端架构
- 能解释关键 trade-off
- 能分析 failure/recovery
- 能把项目变成面试故事

        ## 本周前置检查

        - 前 11 周核心能力完成；不再学习新框架

        ## 本周必须记住的英文术语

        - `system design`：系统设计
- `ADR`：架构决策记录
- `failure injection`：故障注入
- `trade-off`：权衡
- `SLO`：服务目标

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. System Design 的目标不是画很多框

    **是什么（What）**  
    系统设计是把需求、约束、数据流、状态、失败模式与 trade-off 串起来。

    **为什么需要（Why）**  
    面试和生产评审关心为什么这样设计，而不是用了多少框架。

    **什么时候用（When）**  
    Production AI architecture review。

    **常见误区 / 容易踩坑**  
    不要一上来就说 LangGraph/Celery/Kafka；先讲问题和约束。


**中英术语**：`architecture` = 架构；`trade-off` = 权衡；`failure mode` = 失败模式；`ADR` = 架构决策记录

### 最终架构你至少应该能解释这些组件

```text
Client
  ↓
FastAPI
  ↓
Intent / QueryPlan
  ↓
Semantic Layer + Guardrail
  ↓
┌───────────────┬────────────────┐
│ sync query    │ long analysis  │
↓               ↓
PostgreSQL      Queue → Worker
│               │
└──────→ Result / Chart
        ↓
Trace / Metrics / Eval
```

## 2. ADR：为什么选择这个方案？

    **是什么（What）**  
    Architecture Decision Record 用简短结构记录 Context / Decision / Alternatives / Consequences。

    **为什么需要（Why）**  
    帮助你形成工程判断，不是只记最终架构。

    **什么时候用（When）**  
    例如 LLM direct SQL vs semantic layer、sync vs worker、optimistic vs pessimistic locking。

    **常见误区 / 容易踩坑**  
    不要只写“选择 X，因为更好”；必须写代价。


**中英术语**：`ADR` = 架构决策记录；`alternative` = 备选方案；`consequence` = 后果/代价

### 练习：写一条 ADR

主题：为什么 ChatBI 选择 Semantic Layer，而不是 LLM 直接生成并执行任意 SQL。

<details>
    <summary><b>参考答案</b></summary>

    ```python
# Context:
# 用户自然语言灵活，但数据库字段/权限/指标定义必须稳定。

# Decision:
# LLM 输出 QueryPlan；程序通过 Semantic Layer/Allowlist 构造 SQL。

# Alternatives:
# 1. LLM direct SQL
# 2. 固定报表模板

# Consequences:
# + 安全、可解释、一致
# - 需要维护 metadata/semantic layer
# - 新 metric/dimension 需要注册
```

    </details>

## 3. Failure Injection

    **是什么（What）**  
    主动制造故障，验证系统不是只在 happy path 可用。

    **为什么需要（Why）**  
    Production reliability 必须通过实验验证。

    **什么时候用（When）**  
    LLM timeout、worker crash、duplicate POST、DB pool exhausted、invalid plan。

    **常见误区 / 容易踩坑**  
    只写“我们有 retry”不等于真的恢复过。


**中英术语**：`failure injection` = 故障注入；`happy path` = 正常路径；`recovery` = 恢复

### 最终 Failure Matrix

| Failure | 你应该能说出的机制 |
|---|---|
| LLM timeout | timeout + limited retry + fallback |
| duplicate POST | idempotency key + unique constraint |
| worker crash | heartbeat/lease + reconciliation + retry |
| DB pool exhausted | pool limit + timeout + backpressure |
| invalid query plan | schema + allowlist + validator |
| tool 503 | classify transient error + retry/backoff |

### Final Gate

1. 500 concurrent requests 为什么不是 500 DB connections？
2. timeout 为什么不能直接判定失败？
3. checkpoint 为什么不能代替业务数据库？
4. 为什么不让 LLM 任意执行 SQL？
5. worker crash 后如何恢复？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 应用通过有限 connection pool 复用连接；数据库连接是昂贵且有限的资源。

**2.** 远端可能已成功执行，只是响应丢失/延迟；需要幂等和状态确认。

**3.** Checkpoint 服务 workflow resume；业务数据库负责长期事实、一致性、约束和查询。

**4.** 需要权限、metric 定义、字段 allowlist、row limit 和可审计性，这些应由确定性系统控制。

**5.** 用 heartbeat/lease 识别 stale RUNNING，再 reconciliation → retry；任务还必须幂等。

    </details>

## ChatBI 本周作业

        完成：
- 一张端到端架构图
- 3~6 个 Failure Injection
- 3 条 ADR
- README
- 10 分钟项目讲解

### 本周面试 / 理解检查

1. Production AI 项目最重要的三个维度？
2. 为什么 Agentic 不应该覆盖整个系统？
3. 你如何证明系统版本变好了？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 正确性/质量、可靠性、可观测与可恢复性；同时需要性能/成本 trade-off。

**2.** 很多安全、权限、状态、数据规则更适合 deterministic 程序；Agent 只用于模糊理解和需要推理的部分。

**3.** Golden eval + failure tests + performance metrics + trace evidence，而不是主观 demo 感觉。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。